In [ ]:
%cd ..

# TICA

In [ ]:
from dataclasses import dataclass
import pickle
import numpy as np
from deeptime.decomposition import TICA

@dataclass(slots=True, frozen=True)
class NBConfig:
    # TICA settings
    n_components: int
    lagtime: int

In [ ]:
def _compute_tica(data: np.ndarray, config: NBConfig):
    """Compute TICA on the given data."""
    flat_data = data.reshape(data.shape[0], -1)

    tica = TICA(dim=config.n_components, lagtime=config.lagtime)
    tica = tica.fit_fetch(flat_data)

    components = tica.transform(flat_data) # type: ignore

    n_components = components.shape[1]

    singular_values = tica.singular_values[:n_components] # type: ignore
    timescales = tica.timescales(lagtime=config.lagtime)[:n_components] # type: ignore
    vamp2 = (tica.singular_values ** 2).sum() # type: ignore

    stats = {
        "singular_values": singular_values,
        "timescales": timescales,
        "vamp2": float(vamp2)
    }
    return components, stats


def compute_tica(features: dict[str, np.ndarray], config: NBConfig):
    """Compute TICA"""
    results_data = {}
    results_stats = {}
    for representation, data in features.items():
        components, stats = _compute_tica(data, config)
        results_data[representation] = components
        results_stats[representation] = stats
    return results_data, results_stats


In [ ]:
top_pdb = "examples/_structure.pdb"
traj_dcd = "examples/_trajectory.dcd"

config = NBConfig(n_components=2, lagtime=10)

In [ ]:
with open("examples/pcs_preprocessing.pkl", "rb") as h:
    components = pickle.load(h)["data"]

stats = compute_tica(components, config)

with open("examples/tica_2.pkl", "wb") as h:
    pickle.dump({
        "data": components,
        "stats": stats
    }, h, protocol=pickle.HIGHEST_PROTOCOL)